# Project Profiles Workflow

Project profiles let one local workspace hold several independent literature-review projects. This matters because each review can have its own registry, BibTeX file, notes, themes, and reports without mixing evidence between topics.

This notebook uses only the synthetic project data included in the repository.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "paper_workbench").exists():
    root = root.parent
sys.path.insert(0, str(root))

def rel(path):
    return str(Path(path).relative_to(root))

## Inspect Available Profiles

A profile is just a local folder plus `project.json`. Listing profiles first helps confirm which project you are about to audit or report on.

In [ ]:
from paper_workbench.projects import list_project_profiles, load_project_profile

profiles = list_project_profiles(root)
[(profile.name, rel(profile.registry_path), rel(profile.notes_dir), rel(profile.reports_dir)) for profile in profiles]

## Load One Synthetic Project

The `zis_photocatalysis` fixture has a small registry, a BibTeX file, one structured note, and a theme definition. It is intentionally small enough to inspect manually.

In [ ]:
from paper_workbench.bibtex import parse_bibtex_file
from paper_workbench.claims import collect_claims, collect_notes
from paper_workbench.registry import load_registry
from paper_workbench.tags import load_themes

profile = load_project_profile("zis_photocatalysis", root=root)
papers = load_registry(profile.registry_path)
notes = collect_notes(profile.notes_dir)
claims = collect_claims(profile.notes_dir)
entries = parse_bibtex_file(profile.bibtex_path)
themes = load_themes(profile.themes_path)

{
    "project": profile.name,
    "registry": rel(profile.registry_path),
    "papers": len(papers),
    "notes": len(notes),
    "claims": len(claims),
    "bibtex_entries": len(entries),
    "themes": [theme.name for theme in themes],
}

## Check Project Health

Workspace health findings show whether project files line up before you trust generated reports.

In [ ]:
from collections import Counter
from paper_workbench.doctor import workspace_health

health_findings = workspace_health(
    root=profile.root,
    registry_path=profile.registry_path,
    bibtex_path=profile.bibtex_path,
    notes_dir=profile.notes_dir,
    themes_path=profile.themes_path,
    reports_dir=profile.reports_dir,
    profile=profile,
)
Counter(finding.code for finding in health_findings)

## Preview Project Reports Without Writing Files

The same profile data can feed evidence maps and outlines. In a real workflow you would regenerate Markdown reports with the CLI.

In [ ]:
from paper_workbench.reporting import evidence_map_report, section_outline_report

evidence_map = evidence_map_report(papers, claims, themes, notes)
outline = section_outline_report("photocorrosion", papers, claims, themes, notes)
{
    "evidence_map_preview": evidence_map.splitlines()[:12],
    "outline_preview": outline.splitlines()[:12],
}

## Key Takeaways

- A project profile keeps one review's registry, notes, BibTeX, themes, and reports together.
- Profile paths are local files; there is no database or cloud service.
- Run health checks before trusting evidence maps or section outlines.
- Use `--project NAME` in the CLI when you want commands to use profile paths.